# Analiza eksploracyjna zbioru Chest X-rays (Indiana University)
źródło: https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
DATASET_PATH = "../data/iu/"
CSV_NAME1 = "indiana_projections.csv"
CSV_NAME2 = "indiana_reports.csv"

In [ ]:
proj = pd.read_csv(DATASET_PATH+CSV_NAME1, index_col=0)
reports = pd.read_csv(DATASET_PATH+CSV_NAME2, index_col=0)

---

In [ ]:
proj.info()

In [ ]:
proj.head()

In [ ]:
proj.isna().sum()

In [ ]:
counts = proj.groupby('projection')['filename'].count()
counts

In [ ]:
plt.figure(dpi=100)
plt.bar(counts.index, counts.values)
plt.show()

---

In [ ]:
reports.info()

In [ ]:
reports.head()

In [ ]:
missing = reports.isna().sum().sort_values()

In [ ]:
plt.figure(dpi=100)
plt.bar(missing.index,missing.values)
plt.title("Ilość braków w kolumnach")
plt.xticks(rotation=45)
plt.show()

In [ ]:
categorical_columns = list(reports.columns)
categorical_columns.remove("findings")

In [ ]:
d = {}
for cat in categorical_columns:
    d[cat] = reports[cat].value_counts().to_dict()


In [ ]:
import itertools
for k,v in d.items():
    top_5_items = dict(itertools.islice(v.items(), 5))
    if not top_5_items:
        continue
    print(f"{k}: {top_5_items}")
    plt.figure(dpi=100)
    plt.bar(top_5_items.keys(), top_5_items.values())
    plt.xticks(rotation=45)
    plt.title(f"TOP 5 klas w {k}")
    plt.show()

---

### Analiza zmiennej opisywanej: `findings`

In [ ]:
findings = reports['findings'].copy()
findings

In [ ]:
print(findings.isna().sum())

In [ ]:
findings.dropna(inplace=True)

In [ ]:
import re
from collections import Counter
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
all_tokens = []

for text in findings:
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    all_tokens.extend(filtered_tokens)

word_counts = Counter(all_tokens)

In [ ]:
N=10
most_common = word_counts.most_common(N)
words = [item[0] for item in most_common]
counts = [item[1] for item in most_common]

plt.figure(dpi=100)
plt.barh(words, counts,)
plt.title("10 najczęściej występujących słów")
plt.xticks(rotation=45, ha="right")
plt.show()

Niepotrzebnym słowem jest `XXXX` użyte w celu anonimizacji pacjentów. W celu oczyszczenia danych usunę wszystkie jego wystąpienia.

In [ ]:
findings = findings.str.replace(r'\s?XXXX', '', regex=True)

In [ ]:
findings.head(10)

Podsumowanie wykonanych operacji w celu oczyszczenia danych.

In [ ]:
def clean(column):
    cleaned_column = column.copy()
    cleaned_column = cleaned_column.dropna()
    cleaned_column = cleaned_column.str.replace(r'\s?XXXX', '', regex=True)
    return cleaned_column

---

## Doprowadzenie zbioru do postaci [`id`, `obraz`, `raport`]
Zdecydowałem, żeby nie uwzględniać zdjęć bocznych, zbiór będzie uwzględniał jedynie zdjęcia typu `frontal`

In [ ]:
final_csv = proj.copy()
final_csv = final_csv[proj['projection'] == "Frontal"]

In [ ]:
final_csv = final_csv.drop("projection", axis=1)

In [ ]:
final_csv = final_csv.join(findings, on=['uid'])

In [ ]:
final_csv = final_csv.dropna()

In [ ]:
final_csv.info()

In [ ]:
final_csv.to_csv("df.csv")